# Proposed validation — review before it runs

**What this measures:** Two arms from bit-identical init on pi0's mixture-of-transformers at dummy size: under stop_vlm the VLM trunk's weights must move exactly 0 while the action expert's still move and its loss falls; under full routing both move. Reviewer note: the probe runs on gemma.Module (where the routing switch lives) rather than the full Pi0, whose SigLIP tower does not fit the CPU sandbox.

**Target metric:** `vlm_weight_l1_delta_stop_vlm`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [1]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "750c437ae68efb70c6dd52cc337dcb3d62478c4a"
seed = 0


## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_gradient_routing.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [3]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_gradient_routing.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

[remyx] cwd /workspace/target_repo | script /workspace/target_repo/eval/eval_gradient_routing.py


In [4]:
"""Eval: with gradient_routing='stop_vlm', does the VLM stop learning from the action expert
while the action expert keeps learning? (arXiv:2611.05249 insulation factor, dummy-scale probe)

Runs pi0's mixture-of-transformers (openpi.models.gemma.Module, the module where the PR's
gradient_routing switch lives) at the `dummy` size the PR's own tests use: stream 0 is the VLM
(PaliGemma) trunk, stream 1 the action expert. Two arms from bit-identical init on ONE fixed
random batch, `full` vs `stop_vlm`, 20 plain-SGD steps on the action expert's loss (MSE of the
expert stream's output against a fixed target). Reports per-side weight L1 movement (VLM vs
action expert), loss before/after, and the step-0 loss gap. The whole comparison lives inside
this single run (capability question, baseline: none). CPU only, no datasets/checkpoints/network.

Why not the full Pi0 wrapper: Pi0 always builds the So400m/14 SigLIP tower (~400M params); on
CPU that peaks above 10 GB with any optimizer, over the run sandbox's memory. The routing switch
is implemented in gemma.Module (stop_gradient on the trunk's q/k/v after projection), so this
exercises the mechanism itself in under a minute and ~1 GB.

Baseline safety: `gradient_routing` is a kwarg the PR adds to gemma.Module. It is passed only
when the constructor accepts it; on the pre-change baseline both arms fall back to the default
full coupling, the target metric reads nonzero under the "stop_vlm" arm, and the eval fails as it
should. On the PR head the two arms execute genuinely different gradient paths.
"""
import argparse
import inspect
import json
import os
import sys
from pathlib import Path

In [5]:
ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "packages" / "openpi-client" / "src"))

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # CPU-only compute; never measure CUDA quantities
import jax  # noqa: E402

In [6]:
import flax.nnx as nnx  # noqa: E402
import flax.nnx.bridge as nnx_bridge  # noqa: E402
import jax.numpy as jnp  # noqa: E402

import openpi.models.gemma as _gemma  # noqa: E402

LR = 1e-2
STEPS = 2 if os.environ.get("REMYX_SMOKE") == "1" else 20
BATCH, PREFIX_LEN, SUFFIX_LEN = 2, 5, 3

In [7]:
# Knob added by the PR to gemma.Module. Absent on the baseline, where it must NOT be passed
# (unknown kwarg -> TypeError); presence is detected via the constructor signature.
_HAS_GRADIENT_ROUTING = "gradient_routing" in inspect.signature(_gemma.Module.__init__).parameters

# One fixed random batch, keys 1-3 as in the PR's gradient_routing_test.py construction.
WIDTH = _gemma.get_config("dummy").width
PREFIX = jax.random.normal(jax.random.key(1), (BATCH, PREFIX_LEN, WIDTH))   # VLM stream input
SUFFIX = jax.random.normal(jax.random.key(2), (BATCH, SUFFIX_LEN, WIDTH))   # action-expert stream input
TARGET = jax.random.normal(jax.random.key(3), (BATCH, SUFFIX_LEN, WIDTH))   # what the expert must produce
MASK = jnp.ones((BATCH, PREFIX_LEN + SUFFIX_LEN, PREFIX_LEN + SUFFIX_LEN), dtype=jnp.bool_)
POSITIONS = jnp.arange(PREFIX_LEN + SUFFIX_LEN)[None, :].repeat(BATCH, axis=0)

In [8]:
def _build(routing: str):
    config = _gemma.get_config("dummy")
    kwargs = dict(configs=[config, config], embed_dtype="float32")
    if _HAS_GRADIENT_ROUTING:
        # In-script enablement of the changed code path; the baseline falls back to full routing.
        kwargs["gradient_routing"] = routing
    llm = nnx_bridge.ToNNX(_gemma.Module(**kwargs))
    llm.lazy_init(rngs=nnx.Rngs(jax.random.key(0)), method="init", use_adarms=[False, False])
    return llm  # bit-identical init in both arms

In [9]:
def _is_expert(path: str) -> bool:
    # Repo test convention (gradient_routing_test.py): the action expert's stream carries a "_1"
    # name suffix. Everything else is the VLM trunk.
    return "_1" in path

def _run_arm(routing: str) -> dict:
    graphdef, params, rest = nnx.split(_build(routing), nnx.Param, ...)
    before = {"/".join(map(str, p)): jnp.array(v.value) for p, v in params.flat_state().items()}

    def loss_fn(p):
        (_, suffix_out), _ = nnx.merge(graphdef, p, rest)([PREFIX, SUFFIX], positions=POSITIONS, mask=MASK)
        return jnp.mean(jnp.square(suffix_out - TARGET))  # the action expert's loss

    @jax.jit
    def step(p):
        # Plain SGD: no optimizer state, no weight decay, so a zero-grad parameter gets exactly
        # 0.0 update and VLM weights under stop_vlm stay bit-identical.
        loss, grads = jax.value_and_grad(loss_fn)(p)
        return loss, jax.tree_util.tree_map(lambda x, g: x - LR * g, p, grads)

    loss_before = float(loss_fn(params))
    for _ in range(STEPS):
        _, params = step(params)
    loss_after = float(loss_fn(params))

    after = {"/".join(map(str, p)): v.value for p, v in params.flat_state().items()}
    vlm_delta = sum(float(jnp.abs(after[p] - before[p]).sum()) for p in before if not _is_expert(p))
    expert_delta = sum(float(jnp.abs(after[p] - before[p]).sum()) for p in before if _is_expert(p))
    print(f"[{routing}] {STEPS} steps; loss {loss_before:.6f} -> {loss_after:.6f}; "
          f"VLM L1 move {vlm_delta:.6f}; expert L1 move {expert_delta:.6f}", flush=True)
    return {
        "vlm_delta": vlm_delta,
        "expert_delta": expert_delta,
        "vlm_count": int(sum(before[p].size for p in before if not _is_expert(p))),
        "expert_count": int(sum(before[p].size for p in before if _is_expert(p))),
        "loss_before": loss_before,
        "loss_after": loss_after,
    }

In [10]:
def main():
    # Accepted and ignored per the output contract.
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", type=int, default=None)
    parser.parse_args()

    full = _run_arm("full")
    stop = _run_arm("stop_vlm")

    metrics = {
        "vlm_weight_l1_delta_stop_vlm": stop["vlm_delta"],
        "vlm_weight_l1_delta_full": full["vlm_delta"],
        "expert_weight_l1_delta_stop_vlm": stop["expert_delta"],
        "expert_weight_l1_delta_full": full["expert_delta"],
        "loss_drop_full": full["loss_before"] - full["loss_after"],
        "loss_drop_stop_vlm": stop["loss_before"] - stop["loss_after"],
        "loss_step0_gap": abs(full["loss_before"] - stop["loss_before"]),
        "loss_before_full": full["loss_before"],
        "loss_after_full": full["loss_after"],
        "loss_before_stop_vlm": stop["loss_before"],
        "loss_after_stop_vlm": stop["loss_after"],
        "vlm_param_count": full["vlm_count"],
        "expert_param_count": full["expert_count"],
    }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

[full] 20 steps; loss 2.115409 -> 1.192903; VLM L1 move 37.593364; expert L1 move 94.101124


[stop_vlm] 20 steps; loss 2.115409 -> 1.354016; VLM L1 move 0.000000; expert L1 move 97.776431


{"vlm_weight_l1_delta_stop_vlm": 0.0, "vlm_weight_l1_delta_full": 37.5933642834425, "expert_weight_l1_delta_stop_vlm": 97.77643121778965, "expert_weight_l1_delta_full": 94.10112391412258, "loss_drop_full": 0.9225068092346191, "loss_drop_stop_vlm": 0.7613930702209473, "loss_step0_gap": 0.0, "loss_before_full": 2.1154093742370605, "loss_after_full": 1.1929025650024414, "loss_before_stop_vlm": 2.1154093742370605, "loss_after_stop_vlm": 1.3540163040161133, "vlm_param_count": 16630336, "expert_param_count": 172608}


## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "With gradient_routing=stop_vlm, does the VLM stop learning from the action expert while the action expert keeps learning?"
loop:
  max_iterations: 8
  fix_code: true
benchmarks:
  - name: pi0-gradient-routing-insulation
    suite:
      notebook:
        path: "eval/eval_gradient_routing.ipynb"
        # The runner image cannot `pip install -e .` this repo (its torch pin conflicts with the
        # CPU torch the builder pins), so the test imports from src/ and declares its own deps.
        packages: ["jax==0.5.3", "flax==0.10.2", "orbax-checkpoint==0.11.13", "optax", "augmax>=0.3.4", "safetensors", "einops", "jaxtyping==0.2.36", "sentencepiece", "transformers==4.53.2", "filelock", "fsspec", "tqdm-loggable", "beartype==0.19.0", "chex==0.1.90", "equinox>=0.11.8", "dm-tree", "ml_collections==1.0.0", "numpy<2", "numpydantic", "treescope", "typing-extensions", "pillow", "opencv-python-headless", "imageio"]
    scorer: vlm_weight_l1_delta_stop_vlm
    baseline: none
    policy:
      guardrail_veto: true
    metrics:
      - name: vlm_weight_l1_delta_stop_vlm
        direction: min
        threshold: 0.0
        role: target
        bar: goal
        reads_as: "Sum of |weight change| over all VLM-trunk params after 20 SGD steps under stop_vlm; pass requires exactly 0.0"
      - name: vlm_weight_l1_delta_full
        direction: max
        threshold: 1.024  # floor: measured 4.57 after 2 steps and 37.6 after 20 at lr 1e-2; 1.024 fails only a dead trunk gradient path
        role: guardrail
        bar: floor
        reads_as: "Same VLM-trunk L1 movement under default full routing; nonzero proves the expert loss couples into the trunk"
      - name: expert_weight_l1_delta_stop_vlm
        direction: max
        threshold: 5.12  # floor: measured 11.1 after 2 steps and 97.8 after 20 at lr 1e-2; 5.12 fails only a frozen expert
        role: guardrail
        bar: floor
        reads_as: "L1 movement of action-expert params (path contains _1) under stop_vlm; expert must keep training"
      - name: expert_weight_l1_delta_full
        direction: max
        threshold: 5.12  # same floor; the expert trains under both routings (measured 94.1 after 20 steps)
        role: guardrail
        bar: floor
        reads_as: "L1 movement of action-expert params under full routing; the expert moves in both arms per the pass rule"
      - name: loss_drop_full
        direction: max
        threshold: 0.00000015  # floor: float32-resolvable drop; measured 0.92 after 20 steps
        role: guardrail
        bar: floor
        reads_as: "Action-expert loss decrease (before minus after) over 20 steps under full routing"
      - name: loss_drop_stop_vlm
        direction: max
        threshold: 0.00000015  # same floor; insulation must not stall the loss (measured 0.76 after 20 steps)
        role: guardrail
        bar: floor
        reads_as: "Action-expert loss decrease under stop_vlm; the loss must still fall when the trunk is insulated"
      - name: loss_step0_gap
        direction: min
        threshold: 0.0
        role: guardrail
        bar: sanity
        reads_as: "Absolute gap between the two routings' initial losses; stop_gradient is forward-identity so it must be exactly 0"
      - name: loss_before_full
        direction: max
        threshold: 0.5  # MSE of the expert output against a unit-normal target at init ~ 2 (measured 2.12)
        role: diagnostic
        bar: sanity
        reads_as: "Action-expert loss at the shared init, full routing"
      - name: loss_after_full
        direction: max
        threshold: 0.5  # 20 steps at lr 1e-2 leave the loss O(1) (measured 1.19); catches divergence
        role: diagnostic
        bar: sanity
        reads_as: "Action-expert loss after 20 steps, full routing"
      - name: loss_before_stop_vlm
        direction: max
        threshold: 0.5  # identical to the full arm's initial loss by forward-identity
        role: diagnostic
        bar: sanity
        reads_as: "Action-expert loss at the shared init, stop_vlm routing"
      - name: loss_after_stop_vlm
        direction: max
        threshold: 0.5
        role: diagnostic
        bar: sanity
        reads_as: "Action-expert loss after 20 steps, stop_vlm routing"
      - name: vlm_param_count
        direction: max
        threshold: 16000000  # embedder 257152*64 = 16.46M + 4 trunk blocks = 16.63M
        role: diagnostic
        bar: sanity
        reads_as: "VLM-side parameter coordinate count in the partition; verifies the VLM bucket is the big one"
      - name: expert_param_count
        direction: max
        threshold: 100000  # 4 expert blocks = 172,608 coords
        role: diagnostic
        bar: sanity
        reads_as: "Action-expert parameter coordinate count; if ~0 the path-based partition is broken"
compute:
  tier: cpu
  timeout_s: 1800  # both arms run in well under a minute (~3 s per arm on 4 CPUs, ~900 MB peak); the default budget is generous
held_constant:
  - "bit-identical init: both arms from openpi.models.gemma.Module(configs=[dummy, dummy], embed_dtype='float32'), lazy_init with jax.random.key(0) (the PR's gradient_routing_test construction)"
  - "one fixed random batch: prefix (VLM stream) 2x5x64, suffix (action-expert stream) 2x3x64, a fixed 2x3x64 target for the expert's output, full attention mask; keys 1-3"
  - "the action expert's loss: mean squared error of the expert stream's output against the fixed target"
  - "plain SGD lr=1e-2, 20 steps (REMYX_SMOKE=1 -> 2 steps), float32; no optimizer state and no weight decay, so a zero-grad parameter gets exactly 0.0 update"
  - "the PR-only gradient_routing kwarg is passed only when gemma.Module's constructor accepts it; the baseline falls back to pre-change full routing instead of TypeError"
  - "parameter partition by the repo test convention: expert side = flat path contains '_1'; everything else is the VLM trunk"
avoid:
  - "no datasets, checkpoints, or network fetches; both transformer streams use the tiny dummy variant"
  - "do not instantiate the full Pi0 wrapper: it always builds the So400m/14 SigLIP tower (~400M params) and peaks above 10 GB on CPU, over the run sandbox's memory; the routing switch lives in gemma.Module"
  - "no wall-clock timing on shared CPU; all reported quantities are deterministic weight/loss deltas"
  - "never use a stateful optimizer or any weight_decay>0: decay moves zero-grad VLM weights and fakes a nonzero VLM delta under stop_vlm"
  - "never pass the PR-only kwarg unconditionally: on the pre-change baseline that is an unknown-kwarg TypeError, not a clean fallback"
  - "a 20-step dummy-scale run cannot test the paper's training-outcome comparisons; do not read it as one"
  - "JAX backend only; the gradient_routing port in models_pytorch is not exercised here"
provenance:
  vlm_weight_l1_delta_stop_vlm: "user_guidance"
  vlm_weight_l1_delta_full: "user_guidance"
  expert_weight_l1_delta_stop_vlm: "user_guidance"
  expert_weight_l1_delta_full: "user_guidance"
  loss_drop_full: "user_guidance"
  loss_drop_stop_vlm: "user_guidance"
  loss_step0_gap: "inferred"
  loss_before_full: "user_guidance"
  loss_after_full: "user_guidance"
  loss_before_stop_vlm: "user_guidance"
  loss_after_stop_vlm: "user_guidance"
  vlm_param_count: "inferred"
  expert_param_count: "inferred"
  held_constant: "repo test convention from src/openpi/models/gradient_routing_test.py; reviewer moved the probe from the full Pi0 to gemma.Module so it fits the CPU sandbox"
  suite: "synthesized"
  compute: "user_guidance"
report:
  headline: "Flip gradient_routing to stop_vlm and the VLM trunk stops moving while the action expert keeps learning; flip it back and both move"
  findings:
    - "VLM-side L1 movement under stop_vlm is {vlm_weight_l1_delta_stop_vlm} (exactly 0 required) versus {vlm_weight_l1_delta_full} under full coupling"
    - "action-expert L1 movement is {expert_weight_l1_delta_stop_vlm} under stop_vlm and {expert_weight_l1_delta_full} under full"
    - "the action expert's loss falls by {loss_drop_full} under full and {loss_drop_stop_vlm} under stop_vlm, from initial values {loss_before_full} and {loss_before_stop_vlm}"
    - "initial losses agree across routings to {loss_step0_gap}, confirming the routing knob leaves forward outputs unchanged"
    - "the partition covers {vlm_param_count} VLM-side and {expert_param_count} action-expert coordinates"
  establishes:
    - "under stop_vlm the VLM trunk receives exactly zero gradient from the action expert's loss ({vlm_weight_l1_delta_stop_vlm}) while the expert keeps learning ({expert_weight_l1_delta_stop_vlm}, loss drop {loss_drop_stop_vlm})"
    - "under full routing the expert's loss does update the VLM ({vlm_weight_l1_delta_full}), so the knob switches real gradient flow rather than being a no-op"
  does_not_establish:
    - "which routing trains better on real data; the paper's 25%/17%/40% open-loop, closed-loop and manipulation deltas are out of scope at this scale"
    - "the PyTorch backend's gradient_routing port in models_pytorch/pi0_pytorch.py"
    - "action-token supervision, the PR's second knob and the other factor of the paper's 2x2 ablation"
  not_measured:
    - "open-loop trajectory-prediction error, closed-loop route progress, and robot manipulation success rate"
    - "throughput, memory, or wall-clock cost of the routing change"
  caveat: "This exercises pi0's mixture-of-transformers at the 64-wide dummy size on random inputs, leaving the SigLIP tower out (the full Pi0 does not fit a 6 GiB CPU sandbox). The exact VLM zero holds only because this harness has no vision-language or action-token loss term; in the paper's insulated arms the VLM still trains, so this tests the routing mechanism, not the recipe's relative merit."
  next: "Run the paper's 2x2 (gradient routing x action-token supervision) on a real VLA dataset with open- and closed-loop evaluation to compare training outcomes."
```